# Lesson 4 : Hosted Tools

Agent Framework provides a wide variety of built-in tool's object (such as, Code Interpreter, Web Search, File Search, MCP tools, etc), and you can use these useful tools in your agent.

> Note : For available built-in tools, see [API reference](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework).

Especially, the hosted tools in Agent Framework are built-in tools hosted on individual client.  
For instance, ```HostedWebSearchTool``` will seach web to get information - this object is implemented as [OpenAI's web search tool](https://platform.openai.com/docs/guides/tools-web-search) when the client object is ```OpenAIResponsesClient``` or ```AzureAIClient```, it'll use [Bing grounding tool](https://learn.microsoft.com/en-us/python/api/azure-ai-agents/azure.ai.agents.models.binggroundingtool) or [Bing custom search tool](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools-classic/bing-custom-search-samples) when ```AzureAIAgentClient``` (which runs on Azure AI agents), and it'll use [Claude's web search tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool) when ```AnthropicClient```.

In this exercise, we explore hosted tools, including MCP tool's calling.

## Web Search by Hosted Tool

First, we change the example in Lesson 1 to use ```HostedWebSearchTool```.

Same as in Lesson 1, we create a ```AzureAIClient``` object as follows.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
)

Next we build an agent which uses ```HostedWebSearchTool``` as follows, instead of local function's tools.

As I have mentioned above, ```AzureAIClient``` uses OpenAI's web search tool as ```HostedWebSearchTool``` internally.

In [2]:
from agent_framework import ChatAgent
from agent_framework import HostedWebSearchTool

agent = ChatAgent(
    name="WeatherAgentWithSearchTool",
    chat_client=client,
    instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
    tools=[HostedWebSearchTool()])

Now we run the agent as follows.

As you see, this agent knows "what day is it today" or "what is the actual weather condition today", because web search tool is used internally.

In Lesson 1, the tool execution is run on your local environment. In this example, Azure OpenAI Responses API will handle this process on server, because OpenAI's built-in web search tool is used.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka today.")  # "今日の大阪の天気と気温を教えてください。"
display(Markdown(result.text))

Today in **Osaka (Fri, Jan 16, 2026)**: **mostly sunny**.

- **Temperature:** around **15°C / 59°F** high, **~5°C / 41°F** low [Osaka-shi, Osaka, Japan Weather Forecast | AccuWeather](https://www.accuweather.com/en/jp/osaka-shi/225007/weather-forecast/225007)[10 Day Weather - Osaka, Osaka, Japan - The Weather Channel](https://weather.com/weather/tenday/l/Osaka+Osaka+Japan?placeId=441174f51a1951566e6b1d02bd724effab80d7359e5f241540d1aa46dfecc59f)[Osaka, Japan 14 day weather forecast - timeanddate.com](https://www.timeanddate.com/weather/japan/osaka/ext)[Weather - Osaka City - 14-Day Forecast & Rain | Ventusky](https://www.ventusky.com/osaka)[Osaka Weather Forecast](https://www.weather-forecast.com/locations/Osaka/forecasts/latest)

## MCP by Hosted Tool

Next we explore ```HostedMCPTool```.

```HostedMCPTool``` uses MCP tool calling hosted on the client.  
With this tool, MCP server integration in Azure OpenAI Responses API (see [here](https://platform.openai.com/docs/guides/tools-connectors-mcp)) will be used internally.

> Note : On contrary, using built-in ```MCPStdioTool```, ```MCPStreamableHTTPTool```, and ```MCPWebsocketTool```, MCP is handled by the implementation in Agent Framework SDK. (It calls MCP servers to list available tools, and each tool's calling is then handled as a local function in Agent Framework SDK.)  
> This might be useful when MCP (or some part of MCP specification) is not supported in your client (e.g., some remote API models might not support MCP Stdio server tool), because this doesn't depend on the client's capability. (Compared to ```HostedMCPTool```, however, this incurs overhead in your computing environment.)

In this example, we create an agent to answer Microsoft technical questions.  
This agent uses a remote MCP server (Streamable HTTP server), which provides information about Microsoft Learn document.

In [4]:
from agent_framework import HostedMCPTool

agent = ChatAgent(
    name="MSTechKnowledgeAgent",
    chat_client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",  # "あなたは、Microsoft の製品やサービスに関する技術質問に答える Agent です。"
    tools=[HostedMCPTool(
        name="Microsoft Learn MCP",
        url="https://learn.microsoft.com/api/mcp",
        approval_mode="never_require",
    )],
)

Let's ask a technical question about Microsoft Azure.  
In this call, MCP tool calling (about Microsoft Learn document) is used in Azure OpenAI Responses API internally. (Use [tracing](./02_trace.ipynb) and see the internal steps.)

In [5]:
from IPython.display import Markdown, display

result = await agent.run("How to create an Azure storage account using Azure CLI ?")  # "Azure CLI を使って Azure Storage アカウントを作成する方法を教えてください。"
display(Markdown(result.text))

To create an Azure Storage account with the Azure CLI, you typically:

1) **Sign in and pick a subscription**
```bash
az login
az account set --subscription "<SUBSCRIPTION_ID_OR_NAME>"
```

2) **Create a resource group** (skip if you already have one)
```bash
az group create \
  --name <RG_NAME> \
  --location <LOCATION>
```

3) **Create the storage account**
```bash
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --location <LOCATION> \
  --sku Standard_LRS \
  --kind StorageV2
```

### Common options you may want
- **Block public blob access** (recommended in many orgs):
```bash
az storage account create \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --location <LOCATION> \
  --sku Standard_LRS \
  --kind StorageV2 \
  --allow-blob-public-access false
```

- **Require secure transfer (HTTPS only)**:
```bash
az storage account update \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME> \
  --https-only true
```

4) **Verify**
```bash
az storage account show \
  --name <STORAGE_ACCOUNT_NAME> \
  --resource-group <RG_NAME>
```

### Notes / gotchas
- `<STORAGE_ACCOUNT_NAME>` must be **globally unique**, **3–24** chars, **lowercase letters and numbers only**.
- `<LOCATION>` examples: `eastus`, `westus2`, `westeurope`.

If you tell me your intended region, redundancy (LRS/ZRS/GRS), and whether this is for blobs only or general use, I can suggest the best `--sku`/security flags.